# Customer Churn Prediction - Data Preprocessing

## Project Overview

This notebook prepares the IBM Telco Customer Churn dataset for machine learning classification.

The goal of this phase is to transform the raw dataset into a format suitable for training and evaluating classification models.

The preprocessing workflow includes:

- Separating predictive features from the target variables
- Removing unnecessary columns
- Identifying numerical and categorical features
- Splitting the dataset into training and testing sets
- Encoding categorical variables
- Scaling numerical variables
- Building a reusable preprocessing pipeline

All preprocessing transformations will be fitted using the training data to prevent data leakage. 

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

# load the dataset
data_path = Path("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

if not data_path.exists():
    data_path = Path("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df = pd.read_csv(data_path)

# clean TotalCharges using the same logic as EDA
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].astype("string").str.strip(),
    errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(0)

# displace dataset dimensions
print(f"Dataset shape: {df.shape}")

display(df.head())

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Separating Features and Target

The target variable for this classification problem is `Churn`.

The remaining columns will serve as predictive features.

The `customerID` column is removed because it uniquely identifies customers but does not provide meaningful information about their likelihood of churning.

Any temporary target-encoding columns used during EDA are also excluded to prevent accidental target leakage.

- 0 = Customer did not churn
- 1 = Customer churned

This representation will be used throughout model training and evaluation. 

In [2]:
# separate predictive features from the target variable
X = df.drop(
    columns=[
        "customerID",
        "Churn"
    ],
    errors="ignore"
).copy()

# concert target into binary values
y = df["Churn"].map({
    "Yes": 1, 
    "No": 0
})

# verify that the target conversion was successful
print("Feature dataset shape:", X.shape)
print("Target dataset shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentages:")
print((y.value_counts(normalize=True) * 100).round(2))

Feature dataset shape: (7043, 19)
Target dataset shape: (7043,)

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Target percentages:
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64


In [3]:
# inspect feature columns
print("\nFeature columns:")
print(X.columns.tolist())

print("\nFeature data types:")
print(X.dtypes)

# check for missing values
print("\nMissing values in features:")
print(X.isnull().sum())

# verify target values
print("\nUnique target values:")
print(y.unique())

# check for missing target values
print("\nMissing values in target:")
print(y.isnull().sum())


Feature columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']

Feature data types:
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        Float64
dtype: object

Missing values in features:
gender              0
SeniorCitizen       0
Partner             0
Dependents         

## 2. Identifying Numerical and Categorical Features

Before building the preprocessing pipeline, the dataset's predictive features must be separated into numerical and categorical groups.

Numerical features represent measurable quantities and will be standardized using StandardScaler. Standardization places features on a comparable scale, which is particularly useful for algorithms such as Logistic Regression.

Categorical features represent labels or groups rather than continuous measurements. These features will be converted into numerical representations using OneHotEncoder.

Although SeniorCitizen is stored as an integer, it represents a binary category (0 or 1). It will therefore be treated as a categorical feature rather than a continuous numerical measurement. 

Separating these feature types allows us ti apply the appropriate transformations to each group using a ColumnTransformer. 

### 2.1 Feature Type Identification

The `identify_feature_types()` function from   `src/preprocessing.py` is used to automatically identify the numerical and categorical features.

This keeps the feature identification logic separate from the notebook and allows it to be reused in other parts of the project.

In [4]:
# import the preprocessing function
import sys
from pathlib import Path

# locate the project root so the src package can be imported
project_root = Path.cwd().parent

if not (project_root / "src").exists():
    project_root = project_root.parent
    
sys.path.append(str(project_root))

from src.preprocessing import identify_feature_types

# identify feature types
numerical_features, categorical_features = identify_feature_types(X)

# display results
print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nNumber of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))


Numerical features:
['tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Number of numerical features: 3
Number of categorical features: 16


## 3. Building the Preprocessing Pipeline

Machine learning algorithms generally require numerical input. However, the dataset contains both numerical and categorical features, so different transformations must be applied to each group.

A preprocessing pipeline will be created using scikit-learn's Pipeline and ColumnTransformer.

The numerical pipeline will:
- Fill missing values using the median
- Standardize features using StandardScaler

The categorical pipeline will:
- Fill missing values using the most frequent category
- Convert categorical values into binary columns using OneHotEncoder

ColumnTransformer will combine both pipelines and apply the appropriate transformations to their corresponding feature groups.

This approach keeps preprocessing organized, reusable, and consistent across different machine learning models.

The pipeline will be fitted only on the training data after the train/test split to prevent data leakage. 

### 3.1 Creating the Preprocessor

The `build_preprocessor()` function creates the combined ColumnTransformer using the numerical and categorical feature lists identified earlier.

At this stage, we're creating the pipeline but not fitting it to the dataset.

The pipeline will be fitted using training data after the train/test split has been completed. 

In [5]:
# import the preprocessing function
from src.preprocessing import build_preprocessor

# build the preprocessing pipeline
preprocessor = build_preprocessor(
    numerical_features,
    categorical_features
)

# display the pipeline structure
print(preprocessor)

ColumnTransformer(transformers=[('numerical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['gender', 'SeniorCitizen', 'Partner',
                                  'Dependents', 'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection'

## 4. Splitting the Dataset into Training and Testing Sets

Before fitting the preprocessing transformations or training machine learning models, the dataset must be divided into training and testing sets.

An 80.20 split will be used:
- 80% of the records will be used for training
- 20% of the records will be reserved for testing

The training set will be used to fit the preprocessing transformations and train the classification models. The testing set will be used to evaluate model performance on previously unseen customer records.

The train_test_split function from scikit-learn will be used with stratify=y to preserve the original distribution of the target variable in both datasets.

A random_state of 42 ensures that the splits is reproducible, allowing the same records to be assigned to the training and testing sets each time the code is executed.

The preprocessing pipeline will only be fitted on the training data to prevent data leakage.

### 4.1 Creating the Training and Testing Sets

The `split_data()` function from src/preprocessing.py is used to divide the features and target into training and testing datasets.

The same split will be used when comparing classification models, ensuring that each model is evaluated on the same unseen records. 

In [6]:
# import the train/test split function
from src.preprocessing import split_data

# split the dataset
X_train, X_test, y_train, y_test = split_data(X, y)

# display dataset dimensions
print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("\nTraining target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

Training feature shape: (5634, 19)
Testing feature shape: (1409, 19)

Training target shape: (5634,)
Testing target shape: (1409,)


### 4.2 Verifying the Target Distribution

After splitting the dataset, the target distribution is examined in both sets.

Stratification should maintain approximately the same proportion of churned and non-churned customers in the training and testing datasets.

This helps ensure that neither dataset is disproportionately represented by one target class.

In [7]:
# compare target distribution
print("\nTraining target distribution:")
print(y_train.value_counts())
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting target distribution:")
print(y_test.value_counts())
print((y_test.value_counts(normalize=True) * 100).round(2))


Training target distribution:
Churn
0    4139
1    1495
Name: count, dtype: int64
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64

Testing target distribution:
Churn
0    1035
1     374
Name: count, dtype: int64
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64


## 5. Fitting the Preprocessing Pipeline

With the training and testing datasets created, the next step is to apply the preprocessing transformations.

The ColumnTransformer combines the numerical and categorical pipelines created earlier.

The pipeline will be fitted using only the training features. This allows it to learn the numerical scaling parameters and categorical encoding categories without accessing information from the testing dataset.

The fitted pipeline will then transform both datasets:
- Training features will use `fit_transform()`
- Testing features will use `transform()`

This approach prevents data leakage and ensures that both datasets receive consistent transformations.

The resulting datasets contain only numerical values, making them suitable for machine learning algorithms.  

In [8]:
# fit the preprocessing pipeline on training data
X_train_processed = preprocessor.fit_transform(X_train)

# transform testing data using the fitted pipeline
X_test_processed = preprocessor.transform(X_test)

# display transformed dataset dimensions
print("Processed training shape:", X_train_processed.shape)
print("Process testing shape:", X_test_processed)

Processed training shape: (5634, 46)
Process testing shape: [[ 1.60848343  1.62997635  2.70682752 ...  1.          0.
   0.        ]
 [-0.9966836   1.16872526 -0.61026041 ...  1.          0.
   0.        ]
 [ 0.34660565  0.44532428  0.40011607 ...  1.          0.
   0.        ]
 ...
 [-1.11880081 -1.46936546 -0.96787308 ...  0.          0.
   0.        ]
 [ 0.95719167 -1.50088982 -0.54736007 ...  0.          0.
   0.        ]
 [ 1.60848343  0.18317439  1.2577283  ...  0.          0.
   0.        ]]


### 5.1 Inspecting the Transformed Features

After preprocessing, the original categorical columns have been converted into individual binary columns through one-hot encoding.

For example, the Contract feature is separated into categories such as Month-to-month, One year, and Two year.

The numerical features have also been standardized. 

The transformed feature names will be inspected to verify that the encoding and scaling transformations were applied correctly.

In [9]:
# get the names of the transformed features
feature_names = preprocessor.get_feature_names_out()

print("Number of transformed features:", len(feature_names))

print("\nTransformed feature names:")
print(feature_names) 

Number of transformed features: 46

Transformed feature names:
['numerical__tenure' 'numerical__MonthlyCharges' 'numerical__TotalCharges'
 'categorical__gender_Female' 'categorical__gender_Male'
 'categorical__SeniorCitizen_0' 'categorical__SeniorCitizen_1'
 'categorical__Partner_No' 'categorical__Partner_Yes'
 'categorical__Dependents_No' 'categorical__Dependents_Yes'
 'categorical__PhoneService_No' 'categorical__PhoneService_Yes'
 'categorical__MultipleLines_No'
 'categorical__MultipleLines_No phone service'
 'categorical__MultipleLines_Yes' 'categorical__InternetService_DSL'
 'categorical__InternetService_Fiber optic'
 'categorical__InternetService_No' 'categorical__OnlineSecurity_No'
 'categorical__OnlineSecurity_No internet service'
 'categorical__OnlineSecurity_Yes' 'categorical__OnlineBackup_No'
 'categorical__OnlineBackup_No internet service'
 'categorical__OnlineBackup_Yes' 'categorical__DeviceProtection_No'
 'categorical__DeviceProtection_No internet service'
 'categorical__D

In [10]:
from scipy import sparse

# convert a small sample to a dense array for inspection
if sparse.issparse(X_train_processed):
    sample_data = X_train_processed[:5].toarray()
else:
    sample_data = X_train_processed[:5]

# create a dataframe for easier viewing
processed_sample = pd.DataFrame(
    sample_data,
    columns=feature_names
)

display(processed_sample)

,numerical__tenure,numerical__MonthlyCharges,numerical__TotalCharges,categorical__gender_Female,categorical__gender_Male,categorical__SeniorCitizen_0,categorical__SeniorCitizen_1,categorical__Partner_No,categorical__Partner_Yes,categorical__Dependents_No,...,categorical__StreamingMovies_Yes,categorical__Contract_Month-to-month,categorical__Contract_One year,categorical__Contract_Two year,categorical__PaperlessBilling_No,categorical__PaperlessBilling_Yes,categorical__PaymentMethod_Bank transfer (automatic),categorical__PaymentMethod_Credit card (automatic),categorical__PaymentMethod_Electronic check,categorical__PaymentMethod_Mailed check
0,0.102371,-0.521976,-0.262257,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,-0.711743,0.337478,-0.503635,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,-0.793155,-0.809013,-0.749883,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
3,-0.263980,0.284384,-0.172722,1.0,0.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
4,-1.281624,-0.676279,-0.989374,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


### 5.2 Validating the Processed Data

Before moving into model training, the transformed datasets must be checked for consistency and data quality.

The following checks verify that:
- Training and testing datasets have matching feature counts
- No missing values remain after imputation
- All transformed values are finite numerical values
- The preprocessing pipeline generated the expected feature names

These checks help ensure that the processed datasets are ready for machine learning.

In [11]:
import numpy as np

def check_processed_data(data):
    '''
    Check for missing and non-finite values.
    '''
    
    if sparse.issparse(data):
        values = data.data 
    else:
        values = np.asarray(data)
        
    return {
        "Missing values:": np.isnan(values).sum(),
        "Non-finite values": np.isfinite(values).all()
    }
    
# validate training and testing datasets
print("Training data validation:")
print(check_processed_data(X_train_processed))

print("\nTesting data validation:")
print(check_processed_data(X_test_processed))

# verify feature dimensions match
print("\nFeature dimensions match:")
print(X_train_processed.shape[1] == X_test_processed.shape[1])

# very target and feature row counts match
print("\nTraining row counts match:")
print(X_train_processed.shape[0] == len(y_train))

print("\nTesting row counts match:")
print(X_test_processed.shape[0] == len(y_test))

Training data validation:
{'Missing values:': 0, 'Non-finite values': True}

Testing data validation:
{'Missing values:': 0, 'Non-finite values': True}

Feature dimensions match:
True

Training row counts match:
True

Testing row counts match:
True


## 6. Preprocessing Summary

The data preprocessing workflow is complete. The dataset has been prepared for machine learning model development.

The following steps were completed:
1. Separated the features and target variable
2. Identified numerical and categorical features
3. Built a preprocessing pipeline using ColumnTransformer
4. Split the dataset into training and testing sets using stratified sampling
5. Applied imputation, standardization, and one-hot encoding
6. Validated the processed datasets for missing values, non-finite values, and matching dimensions

The final validation checks confirmed that both datasets contain valid numerical values, have matching feature dimensions, and align correctly with their target variables.

The preprocessing pipeline was fitted using only the training data to prevent data leakage.

The processed data is now ready for model training and evaluation. 